# WESAD: pipeline completo y eficiente de extracción de features para clasificación binaria de estrés

Esta notebook implementa un flujo reproducible para:

1. Leer los archivos `.pkl` originales de WESAD.
2. Trabajar únicamente con las señales del dispositivo de pecho RespiBAN.
3. Convertir las etiquetas originales en un target binario configurable.
4. Excluir automáticamente las etiquetas no seleccionadas.
5. Recorrer las señales ventana por ventana sin almacenar todos los segmentos crudos.
6. Extraer features estadísticas, temporales, espectrales, fisiológicas, no lineales, fractales, wavelet y multimodales.
7. Guardar cada sujeto inmediatamente en disco.
8. Liberar memoria después de procesar cada sujeto.
9. Consolidar el resultado final en un diccionario y guardarlo en un archivo `.pkl`.
10. Cargar posteriormente el archivo procesado.

## Decisión principal de eficiencia

No se guardan matrices con forma `(n_ventanas, 42000, canales)`.  
Con ventanas de 60 segundos y un desplazamiento de 250 ms, almacenar los segmentos crudos puede requerir varios gigabytes por sujeto.

La notebook calcula las features directamente sobre cada ventana y conserva solamente:

- matriz de features;
- nombres de las features;
- target binario;
- etiqueta original;
- inicio y fin de la ventana;
- pureza;
- configuración del procesamiento.

## Advertencia computacional

Calcular simultáneamente todas las features avanzadas con:

- ventana de 60 segundos;
- desplazamiento de 250 ms;
- 15 sujetos;

es muy costoso, especialmente en un computador de baja potencia.

Por eso el flujo ofrece:

- procesamiento por sujeto;
- guardado incremental;
- reanudación automática;
- submuestreo interno para features no lineales;
- modo rápido y modo completo;
- activación independiente de familias costosas.

El modo completo conserva todas las familias propuestas, pero puede tardar mucho. Para pruebas iniciales se recomienda aumentar `STEP_MS`.

## Etiquetas y target binario

El protocolo binario predeterminado será:

| Etiqueta original | Estado | Target |
|---:|---|---:|
| 1 | Baseline | 0 |
| 2 | Stress | 1 |
| 3 | Amusement | 0 |
| 0 | Transición o estado no definido | Excluida |
| 4 | Meditation | Excluida por defecto |
| 5, 6, 7 | Fases auxiliares del protocolo | Excluidas |

La configuración es modificable mediante dos listas:

```python
NEGATIVE_LABELS = [1, 3]
POSITIVE_LABELS = [2]
```

Para incluir meditación como no estrés:

```python
NEGATIVE_LABELS = [1, 3, 4]
```

Una ventana solo se acepta si cumple el umbral de pureza definido. Con `PURITY_THRESHOLD = 1.0`, todas sus muestras deben pertenecer a una única etiqueta válida.

## Familias de features incluidas

### ECG

- Estadísticas de amplitud.
- Detección de picos R.
- Frecuencia cardiaca.
- Intervalos RR/NN.
- HRV temporal.
- Poincaré.
- HRV espectral cuando la cantidad de intervalos lo permite.
- Entropía muestral, aproximada, de permutación y Lempel–Ziv.
- DFA.
- Dimensiones fractales de Higuchi, Katz y Petrosian.
- RQA aproximado sobre una versión reducida de la serie RR.

### EDA

- Estadísticas de la señal completa.
- Componentes tónico y fásico.
- Conteo, tasa, amplitud, área y morfología de SCR.
- Entropías.
- Dimensiones fractales.
- Wavelet.
- Features espectrales de muy baja frecuencia.

### EMG

- RMS, MAV, IEMG y longitud de forma de onda.
- Cruces por cero y cambios de signo de pendiente.
- Willison amplitude.
- Teager–Kaiser Energy Operator.
- Frecuencia media, mediana y dominante.
- Momentos espectrales.
- Entropías, fractales y wavelet.

### Respiración

- Frecuencia respiratoria.
- Duración de ciclos.
- Duración de inspiración y espiración.
- Relación inspiración/espiración.
- Variabilidad entre respiraciones.
- Entropía y complejidad.
- Features espectrales y wavelet.

### Temperatura

- Nivel, variabilidad, rango, pendiente y cambio neto.
- Primera y segunda derivada.
- Curvatura.
- Entropías, fractales y wavelet.

### Acelerómetro

Se calculan features para X, Y, Z y magnitud:

- Estadísticas.
- Energía.
- SMA.
- SVM.
- Jerk.
- Ángulos de inclinación.
- Correlaciones entre ejes.
- Entropías.
- Fractales.
- Wavelet.
- Features espectrales.

### Features multimodales

- Correlaciones entre señales.
- Correlación máxima con desfase.
- Información mutua.
- Coherencia espectral.
- Acoplamiento ECG–Resp.
- Acoplamiento EDA–Resp.
- Acoplamiento EMG–ACC.

In [1]:
# Celda opcional de instalación.
# Descomenta y ejecuta una sola vez si alguna librería no está disponible.
# %pip install --upgrade "neurokit2>=0.2.10" "antropy>=0.1.6" "nolds>=0.6.0" "PyWavelets>=1.4" "scipy>=1.10" "numpy>=1.23" "pandas>=1.5" "tqdm>=4.60"

In [2]:
import os
import gc
import json
import math
import pickle
import warnings
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Callable, Tuple, List

import numpy as np
import pandas as pd

from scipy import signal, stats
from scipy.integrate import trapezoid
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy as scipy_entropy
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Dependencias opcionales
try:
    import neurokit2 as nk
    HAS_NEUROKIT = True
except:
    print("no importa neurokit2")

try:
    import pywt
    HAS_PYWT = True
except:
    pass

try:
    import antropy as ant
    HAS_ANTROPY = True
except:
    pass

try:
    import nolds
    HAS_NOLDS = True
except:
    pass

print({
    "NeuroKit2": HAS_NEUROKIT,
    "PyWavelets": HAS_PYWT,
    "AntroPy": HAS_ANTROPY,
    "nolds": HAS_NOLDS
})

{'NeuroKit2': False, 'PyWavelets': True, 'AntroPy': False, 'nolds': False}


## Configuración

Para un computador de baja potencia se recomienda comenzar con:

```python
STEP_MS = 250
```

Cuando el pipeline esté validado, puede cambiarse a `250`.

El parámetro `FULL_FEATURE_SET` activa todas las familias avanzadas.  
Las opciones `ENABLE_RQA`, `ENABLE_WAVELETS`, `ENABLE_FRACTALS` y `ENABLE_MULTIMODAL` permiten desactivar componentes específicos sin modificar el resto del código.

In [3]:
# -----------------------
# Rutas y sujetos
# -----------------------

BASE_PATH = Path.cwd() / "WESAD"
OUTPUT_DIR = Path.cwd() / "wesad_feature_output"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FINAL_OUTPUT_PATH = OUTPUT_DIR / "wesad_processed_full_features.pkl"
CONFIG_PATH = OUTPUT_DIR / "processing_config.json"

SUBJECTS = [
    #"S2", 
    "S3", "S4", "S5",
    "S6", "S7", "S8", "S9",
    "S10", "S11",
    "S13", "S14", "S15", "S16", "S17"
]

# -----------------------
# Target
# -----------------------

NEGATIVE_LABELS = [1, 3]
POSITIVE_LABELS = [2]
PURITY_THRESHOLD = 1.0

# -----------------------
# Ventanas
# -----------------------

WINDOW_MS = 60_000

# Recomendado para un computador de baja potencia durante validación:
STEP_MS = 250

# Para reproducir desplazamiento de 250 ms, cambia a:
# STEP_MS = 250

SAMPLING_RATE = 700.0

SELECTED_SIGNALS = [
    "ACC", "ECG", "EMG", "EDA", "Temp", "Resp"
]

# -----------------------
# Conjunto de features
# -----------------------

FULL_FEATURE_SET = True
ENABLE_WAVELETS = True
ENABLE_FRACTALS = True
ENABLE_ENTROPIES = True
ENABLE_RQA = True
ENABLE_MULTIMODAL = True

# Limita el tamaño de las series usadas en cálculos costosos.
NONLINEAR_MAX_POINTS = 1500
RQA_MAX_POINTS = 400
MUTUAL_INFO_BINS = 32

# Tipo compacto para la salida.
OUTPUT_DTYPE = np.float32

# Guardado y reanudación.
RESUME_FROM_CHECKPOINTS = True
OVERWRITE_EXISTING_SUBJECTS = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "subjects": SUBJECTS,
    "negative_labels": NEGATIVE_LABELS,
    "positive_labels": POSITIVE_LABELS,
    "purity_threshold": PURITY_THRESHOLD,
    "window_ms": WINDOW_MS,
    "step_ms": STEP_MS,
    "sampling_rate": SAMPLING_RATE,
    "selected_signals": SELECTED_SIGNALS,
    "full_feature_set": FULL_FEATURE_SET,
    "enable_wavelets": ENABLE_WAVELETS,
    "enable_fractals": ENABLE_FRACTALS,
    "enable_entropies": ENABLE_ENTROPIES,
    "enable_rqa": ENABLE_RQA,
    "enable_multimodal": ENABLE_MULTIMODAL,
    "nonlinear_max_points": NONLINEAR_MAX_POINTS,
    "rqa_max_points": RQA_MAX_POINTS,
    "mutual_info_bins": MUTUAL_INFO_BINS,
    "output_dtype": str(OUTPUT_DTYPE)
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

CONFIG

{'subjects': ['S3',
  'S4',
  'S5',
  'S6',
  'S7',
  'S8',
  'S9',
  'S10',
  'S11',
  'S13',
  'S14',
  'S15',
  'S16',
  'S17'],
 'negative_labels': [1, 3],
 'positive_labels': [2],
 'purity_threshold': 1.0,
 'window_ms': 60000,
 'step_ms': 250,
 'sampling_rate': 700.0,
 'selected_signals': ['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'],
 'full_feature_set': True,
 'enable_wavelets': True,
 'enable_fractals': True,
 'enable_entropies': True,
 'enable_rqa': True,
 'enable_multimodal': True,
 'nonlinear_max_points': 1500,
 'rqa_max_points': 400,
 'mutual_info_bins': 32,
 'output_dtype': "<class 'numpy.float32'>"}

## Carga eficiente de un sujeto

En lugar de cargar todos los sujetos simultáneamente, se carga uno, se procesa, se guarda y se elimina de memoria.

Esto reduce drásticamente el consumo de RAM.

In [4]:
def load_wesad_subject(base_path: Path, subject: str) -> Dict:
    file_path = base_path / subject / f"{subject}.pkl"

    if not file_path.exists():
        raise FileNotFoundError(f"No se encontró: {file_path}")

    with open(file_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    if "signal" not in data or "chest" not in data["signal"]:
        raise KeyError(f"{subject} no contiene signal/chest")

    if "label" not in data:
        raise KeyError(f"{subject} no contiene label")

    return data


def inspect_subject_structure(base_path: Path, subject: str = "S2") -> None:
    data = load_wesad_subject(base_path, subject)

    print("Claves principales:", data.keys())
    print("Claves signal:", data["signal"].keys())
    print("Claves chest:", data["signal"]["chest"].keys())
    print("Labels:", np.unique(data["label"], return_counts=True))

    for name, values in data["signal"]["chest"].items():
        print(name, np.asarray(values).shape, np.asarray(values).dtype)

    del data
    gc.collect()


# Ejecuta esta celda para verificar la estructura.
inspect_subject_structure(BASE_PATH, "S2")

Claves principales: dict_keys(['signal', 'label', 'subject'])
Claves signal: dict_keys(['chest', 'wrist'])
Claves chest: dict_keys(['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'])
Labels: (array([0, 1, 2, 3, 4, 6, 7], dtype=int32), array([2142701,  800800,  430500,  253400,  537599,   45500,   44800]))
ACC (4255300, 3) float64
ECG (4255300, 1) float64
EMG (4255300, 1) float64
EDA (4255300, 1) float64
Temp (4255300, 1) float32
Resp (4255300, 1) float64


## Funciones auxiliares

Las siguientes funciones:

- limpian valores no finitos;
- reducen series antes de análisis no lineales;
- calculan estadísticas robustas;
- calculan features espectrales;
- evitan que un fallo puntual detenga el procesamiento completo.

In [5]:
EPSILON = 1e-12


def sanitize_signal(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64).reshape(-1)

    if x.size == 0:
        return x

    finite = np.isfinite(x)

    if finite.all():
        return x

    if not finite.any():
        return np.zeros_like(x)

    idx = np.arange(x.size)
    x[~finite] = np.interp(idx[~finite], idx[finite], x[finite])
    return x


def downsample_for_nonlinear(x: np.ndarray, max_points: int = NONLINEAR_MAX_POINTS) -> np.ndarray:
    x = sanitize_signal(x)

    if x.size <= max_points:
        return x

    idx = np.linspace(0, x.size - 1, max_points).astype(np.int64)
    return x[idx]


def safe_float(value, default=np.nan) -> float:
    try:
        value = float(value)
        return value if np.isfinite(value) else default
    except Exception:
        return default


def safe_skew(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    if x.size < 3 or np.std(x) < EPSILON:
        return 0.0
    return safe_float(stats.skew(x, bias=False), 0.0)


def safe_kurtosis(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    if x.size < 4 or np.std(x) < EPSILON:
        return 0.0
    return safe_float(stats.kurtosis(x, fisher=True, bias=False), 0.0)


def linear_slope(x: np.ndarray, sampling_rate: float) -> float:
    x = sanitize_signal(x)
    if x.size < 2:
        return 0.0
    t = np.arange(x.size, dtype=np.float64) / sampling_rate
    return safe_float(np.polyfit(t, x, 1)[0], 0.0)


def zero_crossing_rate(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    if x.size < 2:
        return 0.0
    centered = x - np.mean(x)
    return float(np.mean(centered[:-1] * centered[1:] < 0))


def mean_crossing_rate(x: np.ndarray) -> float:
    return zero_crossing_rate(x)


def waveform_length(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    return float(np.sum(np.abs(np.diff(x)))) if x.size > 1 else 0.0


def slope_sign_changes(x: np.ndarray, threshold: float = 0.0) -> float:
    x = sanitize_signal(x)
    if x.size < 3:
        return 0.0
    d1 = np.diff(x)
    changes = ((d1[:-1] * d1[1:]) < 0) & (
        (np.abs(d1[:-1]) >= threshold) | (np.abs(d1[1:]) >= threshold)
    )
    return float(np.sum(changes))


def willison_amplitude(x: np.ndarray, threshold: Optional[float] = None) -> float:
    x = sanitize_signal(x)
    if x.size < 2:
        return 0.0
    if threshold is None:
        threshold = 0.1 * np.std(x)
    return float(np.sum(np.abs(np.diff(x)) > threshold))


def teager_kaiser_energy(x: np.ndarray) -> Tuple[float, float]:
    x = sanitize_signal(x)
    if x.size < 3:
        return 0.0, 0.0
    tkeo = x[1:-1] ** 2 - x[:-2] * x[2:]
    return safe_float(np.mean(tkeo), 0.0), safe_float(np.std(tkeo), 0.0)


def spectral_features(
    x: np.ndarray,
    sampling_rate: float,
    prefix: str = "",
    bands: Optional[Dict[str, Tuple[float, float]]] = None
) -> Dict[str, float]:
    x = sanitize_signal(x)

    base = {
        f"{prefix}dominant_frequency": 0.0,
        f"{prefix}mean_frequency": 0.0,
        f"{prefix}median_frequency": 0.0,
        f"{prefix}spectral_entropy": 0.0,
        f"{prefix}total_spectral_power": 0.0,
        f"{prefix}spectral_centroid": 0.0,
        f"{prefix}spectral_spread": 0.0,
        f"{prefix}spectral_skewness": 0.0,
        f"{prefix}spectral_kurtosis": 0.0,
    }

    if x.size < 8 or np.std(x) < EPSILON:
        if bands:
            for band_name in bands:
                base[f"{prefix}power_{band_name}"] = 0.0
                base[f"{prefix}relative_power_{band_name}"] = 0.0
        return base

    f, psd = signal.welch(
        x,
        fs=sampling_rate,
        nperseg=min(2048, x.size)
    )

    psd = np.maximum(psd, 0.0)
    psd_sum = np.sum(psd)
    total_power = trapezoid(psd, f) if f.size > 1 else 0.0

    if psd_sum <= EPSILON:
        return base

    p = psd / psd_sum
    dom = f[int(np.argmax(psd))]
    mean_f = np.sum(f * psd) / psd_sum
    cum = np.cumsum(psd)
    median_f = f[min(np.searchsorted(cum, cum[-1] / 2), f.size - 1)]
    spec_entropy = -np.sum(p * np.log2(p + EPSILON)) / np.log2(max(len(p), 2))
    spread = np.sqrt(np.sum(((f - mean_f) ** 2) * p))

    if spread > EPSILON:
        skew = np.sum(((f - mean_f) / spread) ** 3 * p)
        kurt = np.sum(((f - mean_f) / spread) ** 4 * p)
    else:
        skew = 0.0
        kurt = 0.0

    out = {
        f"{prefix}dominant_frequency": safe_float(dom, 0.0),
        f"{prefix}mean_frequency": safe_float(mean_f, 0.0),
        f"{prefix}median_frequency": safe_float(median_f, 0.0),
        f"{prefix}spectral_entropy": safe_float(spec_entropy, 0.0),
        f"{prefix}total_spectral_power": safe_float(total_power, 0.0),
        f"{prefix}spectral_centroid": safe_float(mean_f, 0.0),
        f"{prefix}spectral_spread": safe_float(spread, 0.0),
        f"{prefix}spectral_skewness": safe_float(skew, 0.0),
        f"{prefix}spectral_kurtosis": safe_float(kurt, 0.0),
    }

    if bands:
        for band_name, (low, high) in bands.items():
            mask = (f >= low) & (f < high)
            power = trapezoid(psd[mask], f[mask]) if np.sum(mask) > 1 else 0.0
            out[f"{prefix}power_{band_name}"] = safe_float(power, 0.0)
            out[f"{prefix}relative_power_{band_name}"] = safe_float(
                power / (total_power + EPSILON), 0.0
            )

    return out


def general_features(
    x: np.ndarray,
    sampling_rate: float,
    prefix: str = ""
) -> Dict[str, float]:
    x = sanitize_signal(x)

    if x.size == 0:
        return {}

    q01, q05, q10, q25, q50, q75, q90, q95, q99 = np.percentile(
        x, [1, 5, 10, 25, 50, 75, 90, 95, 99]
    )

    mean = np.mean(x)
    std = np.std(x)
    abs_x = np.abs(x)

    out = {
        f"{prefix}mean": mean,
        f"{prefix}median": q50,
        f"{prefix}std": std,
        f"{prefix}variance": np.var(x),
        f"{prefix}min": np.min(x),
        f"{prefix}max": np.max(x),
        f"{prefix}range": np.ptp(x),
        f"{prefix}q01": q01,
        f"{prefix}q05": q05,
        f"{prefix}q10": q10,
        f"{prefix}q25": q25,
        f"{prefix}q75": q75,
        f"{prefix}q90": q90,
        f"{prefix}q95": q95,
        f"{prefix}q99": q99,
        f"{prefix}iqr": q75 - q25,
        f"{prefix}mad_mean": np.mean(np.abs(x - mean)),
        f"{prefix}mad_median": stats.median_abs_deviation(x),
        f"{prefix}rms": np.sqrt(np.mean(x ** 2)),
        f"{prefix}mean_absolute_value": np.mean(abs_x),
        f"{prefix}energy": np.sum(x ** 2),
        f"{prefix}mean_power": np.mean(x ** 2),
        f"{prefix}auc_absolute": trapezoid(abs_x, dx=1 / sampling_rate),
        f"{prefix}auc_signed": trapezoid(x, dx=1 / sampling_rate),
        f"{prefix}skewness": safe_skew(x),
        f"{prefix}kurtosis": safe_kurtosis(x),
        f"{prefix}slope": linear_slope(x, sampling_rate),
        f"{prefix}delta": x[-1] - x[0],
        f"{prefix}coefficient_variation": std / (abs(mean) + EPSILON),
        f"{prefix}zero_crossing_rate": zero_crossing_rate(x),
        f"{prefix}mean_crossing_rate": mean_crossing_rate(x),
        f"{prefix}waveform_length": waveform_length(x),
        f"{prefix}crest_factor": np.max(abs_x) / (np.sqrt(np.mean(x ** 2)) + EPSILON),
        f"{prefix}shape_factor": np.sqrt(np.mean(x ** 2)) / (np.mean(abs_x) + EPSILON),
        f"{prefix}impulse_factor": np.max(abs_x) / (np.mean(abs_x) + EPSILON),
        f"{prefix}clearance_factor": np.max(abs_x) / (np.mean(np.sqrt(abs_x + EPSILON)) ** 2 + EPSILON),
    }

    return {k: safe_float(v, 0.0) for k, v in out.items()}

## Entropías, fractales, DFA, Lempel–Ziv y RQA

Estas features describen regularidad, complejidad, auto-similitud y estructura dinámica.

Para evitar costos excesivos:

- se usan versiones submuestreadas;
- RQA se calcula sobre un máximo de puntos;
- cualquier fallo devuelve `NaN`, sin detener el sujeto completo.

In [6]:
def approximate_entropy(x: np.ndarray, m: int = 2, r_ratio: float = 0.2) -> float:
    x = downsample_for_nonlinear(x)
    n = len(x)

    if n < m + 2 or np.std(x) < EPSILON:
        return 0.0

    r = r_ratio * np.std(x)

    def _phi(mm):
        patterns = np.array([x[i:i + mm] for i in range(n - mm + 1)])
        distances = np.max(np.abs(patterns[:, None, :] - patterns[None, :, :]), axis=2)
        counts = np.mean(distances <= r, axis=0)
        return np.mean(np.log(counts + EPSILON))

    return safe_float(_phi(m) - _phi(m + 1), np.nan)


def sample_entropy_feature(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < 20 or np.std(x) < EPSILON:
        return 0.0

    try:
        if HAS_ANTROPY:
            return safe_float(ant.sample_entropy(x), np.nan)
        if HAS_NEUROKIT:
            value, _ = nk.entropy_sample(x)
            return safe_float(value, np.nan)
    except Exception:
        pass

    return np.nan


def permutation_entropy_feature(x: np.ndarray, order: int = 3, delay: int = 1) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < order * delay + 2:
        return 0.0

    try:
        if HAS_ANTROPY:
            return safe_float(
                ant.perm_entropy(x, order=order, delay=delay, normalize=True),
                np.nan
            )
    except Exception:
        pass

    patterns = {}
    for i in range(x.size - delay * (order - 1)):
        window = x[i:i + delay * order:delay]
        key = tuple(np.argsort(window))
        patterns[key] = patterns.get(key, 0) + 1

    counts = np.asarray(list(patterns.values()), dtype=np.float64)
    probs = counts / counts.sum()
    pe = -np.sum(probs * np.log2(probs + EPSILON))
    return safe_float(pe / np.log2(math.factorial(order)), np.nan)


def lempel_ziv_complexity_feature(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)
    if x.size < 4:
        return 0.0

    binary = (x > np.median(x)).astype(np.uint8)
    s = "".join(binary.astype(str))

    i, k, l, c, k_max = 0, 1, 1, 1, 1
    n = len(s)

    while True:
        if i + k >= n or l + k >= n:
            c += 1
            break
        if s[i + k] == s[l + k]:
            k += 1
            if l + k >= n:
                c += 1
                break
        else:
            if k > k_max:
                k_max = k
            i += 1
            if i == l:
                c += 1
                l += k_max
                if l + 1 >= n:
                    break
                i, k, k_max = 0, 1, 1
            else:
                k = 1

    normalization = n / max(np.log2(n), EPSILON)
    return safe_float(c / normalization, np.nan)


def petrosian_fd(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)
    if x.size < 3:
        return 0.0
    diff = np.diff(x)
    n_delta = np.sum(diff[1:] * diff[:-1] < 0)
    n = x.size
    denominator = np.log10(n) + np.log10(n / (n + 0.4 * n_delta + EPSILON))
    return safe_float(np.log10(n) / (denominator + EPSILON), np.nan)


def katz_fd(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)
    if x.size < 3:
        return 0.0
    distances = np.abs(np.diff(x))
    length = np.sum(distances)
    diameter = np.max(np.abs(x - x[0]))
    n = x.size
    if length <= EPSILON or diameter <= EPSILON:
        return 1.0
    return safe_float(
        np.log10(n) / (np.log10(n) + np.log10(diameter / length + EPSILON)),
        np.nan
    )


def higuchi_fd(x: np.ndarray, kmax: int = 8) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < 20:
        return np.nan

    if HAS_ANTROPY:
        try:
            return safe_float(ant.higuchi_fd(x, kmax=kmax), np.nan)
        except Exception:
            pass

    n = len(x)
    lengths = []

    for k in range(1, kmax + 1):
        lk = []
        for m in range(k):
            idx = np.arange(1, int(np.floor((n - m - 1) / k)) + 1)
            if idx.size == 0:
                continue
            distance = np.sum(np.abs(x[m + idx * k] - x[m + (idx - 1) * k]))
            norm = (n - 1) / (idx.size * k)
            lk.append(distance * norm / k)
        lengths.append(np.mean(lk) if lk else np.nan)

    lengths = np.asarray(lengths)
    valid = np.isfinite(lengths) & (lengths > 0)
    if np.sum(valid) < 2:
        return np.nan

    return safe_float(
        np.polyfit(np.log(1.0 / np.arange(1, kmax + 1)[valid]), np.log(lengths[valid]), 1)[0],
        np.nan
    )


def dfa_feature(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < 32 or np.std(x) < EPSILON:
        return np.nan

    try:
        if HAS_NOLDS:
            return safe_float(nolds.dfa(x), np.nan)
        if HAS_ANTROPY:
            return safe_float(ant.detrended_fluctuation(x), np.nan)
    except Exception:
        pass

    return np.nan


def rqa_features(x: np.ndarray, max_points: int = RQA_MAX_POINTS) -> Dict[str, float]:
    x = downsample_for_nonlinear(x, max_points=max_points)

    defaults = {
        "rqa_recurrence_rate": np.nan,
        "rqa_determinism": np.nan,
        "rqa_laminarity": np.nan,
        "rqa_diagonal_entropy": np.nan,
    }

    if x.size < 30 or np.std(x) < EPSILON:
        return defaults

    try:
        z = (x - np.mean(x)) / (np.std(x) + EPSILON)
        distances = squareform(pdist(z[:, None], metric="euclidean"))
        epsilon = np.quantile(distances, 0.10)
        recurrence = distances <= epsilon
        np.fill_diagonal(recurrence, False)

        rr = np.mean(recurrence)

        diag_lengths = []
        for offset in range(-len(x) + 1, len(x)):
            diag = np.diag(recurrence, k=offset).astype(np.int8)
            if diag.size < 2:
                continue
            padded = np.r_[0, diag, 0]
            starts = np.where(np.diff(padded) == 1)[0]
            ends = np.where(np.diff(padded) == -1)[0]
            diag_lengths.extend((ends - starts).tolist())

        vert_lengths = []
        for col in range(recurrence.shape[1]):
            colv = recurrence[:, col].astype(np.int8)
            padded = np.r_[0, colv, 0]
            starts = np.where(np.diff(padded) == 1)[0]
            ends = np.where(np.diff(padded) == -1)[0]
            vert_lengths.extend((ends - starts).tolist())

        diag_lengths = np.asarray(diag_lengths)
        vert_lengths = np.asarray(vert_lengths)

        det = (
            np.sum(diag_lengths[diag_lengths >= 2]) / (np.sum(diag_lengths) + EPSILON)
            if diag_lengths.size else 0.0
        )

        lam = (
            np.sum(vert_lengths[vert_lengths >= 2]) / (np.sum(vert_lengths) + EPSILON)
            if vert_lengths.size else 0.0
        )

        valid_diag = diag_lengths[diag_lengths >= 2]
        if valid_diag.size:
            counts = np.bincount(valid_diag)
            probs = counts[counts > 0] / np.sum(counts)
            diag_entropy = -np.sum(probs * np.log(probs + EPSILON))
        else:
            diag_entropy = 0.0

        return {
            "rqa_recurrence_rate": safe_float(rr),
            "rqa_determinism": safe_float(det),
            "rqa_laminarity": safe_float(lam),
            "rqa_diagonal_entropy": safe_float(diag_entropy),
        }

    except Exception:
        return defaults


def nonlinear_features(x: np.ndarray, prefix: str = "") -> Dict[str, float]:
    out = {}

    if ENABLE_ENTROPIES:
        out.update({
            f"{prefix}sample_entropy": sample_entropy_feature(x),
            f"{prefix}approximate_entropy": approximate_entropy(x),
            f"{prefix}permutation_entropy": permutation_entropy_feature(x),
            f"{prefix}lempel_ziv_complexity": lempel_ziv_complexity_feature(x),
        })

    if ENABLE_FRACTALS:
        out.update({
            f"{prefix}petrosian_fd": petrosian_fd(x),
            f"{prefix}katz_fd": katz_fd(x),
            f"{prefix}higuchi_fd": higuchi_fd(x),
            f"{prefix}dfa": dfa_feature(x),
        })

    if ENABLE_RQA:
        out.update({
            f"{prefix}{key}": value
            for key, value in rqa_features(x).items()
        })

    return out

## Wavelet

La transformada wavelet representa cómo se distribuye la energía en diferentes escalas temporales.

Se calculan por nivel:

- energía;
- energía relativa;
- media absoluta;
- desviación estándar;
- entropía de coeficientes.

In [7]:
def wavelet_features(
    x: np.ndarray,
    prefix: str = "",
    wavelet: str = "db4",
    level: int = 4
) -> Dict[str, float]:
    if not ENABLE_WAVELETS or not HAS_PYWT:
        return {}

    x = sanitize_signal(x)

    if x.size < 32:
        return {}

    try:
        max_level = pywt.dwt_max_level(x.size, pywt.Wavelet(wavelet).dec_len)
        use_level = max(1, min(level, max_level))
        coeffs = pywt.wavedec(x, wavelet=wavelet, level=use_level)

        energies = np.asarray([np.sum(c ** 2) for c in coeffs], dtype=np.float64)
        total = np.sum(energies) + EPSILON

        out = {}
        for i, c in enumerate(coeffs):
            label = "A" if i == 0 else f"D{use_level - i + 1}"
            energy = energies[i]
            p = (c ** 2) / (np.sum(c ** 2) + EPSILON)
            coeff_entropy = -np.sum(p * np.log2(p + EPSILON))

            out[f"{prefix}wavelet_{label}_energy"] = safe_float(energy, 0.0)
            out[f"{prefix}wavelet_{label}_relative_energy"] = safe_float(energy / total, 0.0)
            out[f"{prefix}wavelet_{label}_mean_abs"] = safe_float(np.mean(np.abs(c)), 0.0)
            out[f"{prefix}wavelet_{label}_std"] = safe_float(np.std(c), 0.0)
            out[f"{prefix}wavelet_{label}_entropy"] = safe_float(coeff_entropy, 0.0)

        return out
    except Exception:
        return {}

## ECG

Las features de ECG se dividen en:

1. señal cruda;
2. detección de picos R;
3. frecuencia cardiaca;
4. intervalos RR;
5. HRV temporal;
6. Poincaré;
7. HRV espectral;
8. complejidad no lineal de RR.

In [8]:
def extract_ecg_features(ecg: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    ecg = sanitize_signal(ecg)

    out = {}
    out.update(general_features(ecg, sampling_rate, prefix="ecg_raw_"))
    out.update(spectral_features(
        ecg,
        sampling_rate,
        prefix="ecg_raw_",
        bands={
            "very_low": (0.01, 0.5),
            "low": (0.5, 5.0),
            "mid": (5.0, 15.0),
            "high": (15.0, 40.0),
        }
    ))
    out.update(wavelet_features(ecg, prefix="ecg_raw_"))

    if not HAS_NEUROKIT:
        return out

    try:
        cleaned = nk.ecg_clean(ecg, sampling_rate=sampling_rate)
        _, info = nk.ecg_peaks(cleaned, sampling_rate=sampling_rate)
        rpeaks = np.asarray(info.get("ECG_R_Peaks", []), dtype=np.int64)

        out["ecg_rpeak_count"] = float(rpeaks.size)
        out["ecg_rpeak_rate_per_minute"] = float(
            rpeaks.size / (ecg.size / sampling_rate / 60.0 + EPSILON)
        )

        if rpeaks.size >= 3:
            rr_ms = np.diff(rpeaks) / sampling_rate * 1000.0
            hr = 60000.0 / rr_ms
            rr_diff = np.diff(rr_ms)

            out.update({
                "ecg_hr_mean": np.mean(hr),
                "ecg_hr_median": np.median(hr),
                "ecg_hr_std": np.std(hr),
                "ecg_hr_min": np.min(hr),
                "ecg_hr_max": np.max(hr),
                "ecg_hr_range": np.ptp(hr),
                "ecg_mean_nn": np.mean(rr_ms),
                "ecg_median_nn": np.median(rr_ms),
                "ecg_sdnn": np.std(rr_ms, ddof=1) if rr_ms.size > 1 else 0.0,
                "ecg_rmssd": np.sqrt(np.mean(rr_diff ** 2)) if rr_diff.size else 0.0,
                "ecg_sdsd": np.std(rr_diff, ddof=1) if rr_diff.size > 1 else 0.0,
                "ecg_pnn20": np.mean(np.abs(rr_diff) > 20) * 100 if rr_diff.size else 0.0,
                "ecg_pnn50": np.mean(np.abs(rr_diff) > 50) * 100 if rr_diff.size else 0.0,
                "ecg_rr_range": np.ptp(rr_ms),
                "ecg_rr_iqr": np.percentile(rr_ms, 75) - np.percentile(rr_ms, 25),
                "ecg_rr_cv": np.std(rr_ms) / (np.mean(rr_ms) + EPSILON),
                "ecg_rr_mad": stats.median_abs_deviation(rr_ms),
            })

            if rr_diff.size:
                sd1 = np.sqrt(0.5 * np.var(rr_diff))
                sd2 = np.sqrt(max(2 * np.var(rr_ms) - 0.5 * np.var(rr_diff), 0.0))
                out.update({
                    "ecg_sd1": sd1,
                    "ecg_sd2": sd2,
                    "ecg_sd1_sd2_ratio": sd1 / (sd2 + EPSILON),
                    "ecg_poincare_area": np.pi * sd1 * sd2,
                })

            out.update(nonlinear_features(rr_ms, prefix="ecg_rr_"))

            # HRV espectral sobre intervalos interpolados.
            if rr_ms.size >= 10:
                rr_time = np.cumsum(rr_ms) / 1000.0
                interp_fs = 4.0
                uniform_t = np.arange(rr_time[0], rr_time[-1], 1 / interp_fs)
                if uniform_t.size >= 16:
                    interp_rr = np.interp(uniform_t, rr_time, rr_ms)
                    hrv_spec = spectral_features(
                        interp_rr,
                        interp_fs,
                        prefix="ecg_hrv_",
                        bands={
                            "vlf": (0.0033, 0.04),
                            "lf": (0.04, 0.15),
                            "hf": (0.15, 0.40),
                        }
                    )
                    out.update(hrv_spec)
                    lf = out.get("ecg_hrv_power_lf", 0.0)
                    hf = out.get("ecg_hrv_power_hf", 0.0)
                    out["ecg_hrv_lf_hf_ratio"] = lf / (hf + EPSILON)

        return {k: safe_float(v) for k, v in out.items()}

    except Exception:
        return {k: safe_float(v) for k, v in out.items()}

## EDA

Se separan:

- componente tónico: nivel lento de conductancia;
- componente fásico: respuestas rápidas SCR.

También se extraen características morfológicas de cada respuesta.

In [9]:
def extract_eda_features(eda: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    eda = sanitize_signal(eda)

    out = {}
    out.update(general_features(eda, sampling_rate, prefix="eda_raw_"))
    out.update(spectral_features(
        eda,
        sampling_rate,
        prefix="eda_raw_",
        bands={
            "ultra_slow": (0.0, 0.045),
            "slow": (0.045, 0.15),
            "fast": (0.15, 0.5),
            "very_fast": (0.5, 1.0),
        }
    ))
    out.update(nonlinear_features(eda, prefix="eda_raw_"))
    out.update(wavelet_features(eda, prefix="eda_raw_"))

    if not HAS_NEUROKIT:
        return out

    try:
        cleaned = nk.eda_clean(eda, sampling_rate=sampling_rate)
        decomposed = nk.eda_phasic(cleaned, sampling_rate=sampling_rate)

        tonic = sanitize_signal(decomposed["EDA_Tonic"].to_numpy())
        phasic = sanitize_signal(decomposed["EDA_Phasic"].to_numpy())

        out.update(general_features(tonic, sampling_rate, prefix="eda_tonic_"))
        out.update(general_features(phasic, sampling_rate, prefix="eda_phasic_"))
        out.update(nonlinear_features(tonic, prefix="eda_tonic_"))
        out.update(nonlinear_features(phasic, prefix="eda_phasic_"))
        out.update(wavelet_features(phasic, prefix="eda_phasic_"))

        _, info = nk.eda_peaks(phasic, sampling_rate=sampling_rate)

        peaks = np.asarray(info.get("SCR_Peaks", []), dtype=np.int64)
        onsets = np.asarray(info.get("SCR_Onsets", []), dtype=np.int64)
        amplitudes = np.asarray(info.get("SCR_Amplitude", []), dtype=np.float64)
        rise_times = np.asarray(info.get("SCR_RiseTime", []), dtype=np.float64)
        recovery_times = np.asarray(info.get("SCR_RecoveryTime", []), dtype=np.float64)

        amplitudes = amplitudes[np.isfinite(amplitudes)]
        rise_times = rise_times[np.isfinite(rise_times)]
        recovery_times = recovery_times[np.isfinite(recovery_times)]

        duration_min = eda.size / sampling_rate / 60.0

        out.update({
            "eda_scr_count": float(peaks.size),
            "eda_scr_rate_per_minute": peaks.size / (duration_min + EPSILON),
            "eda_scr_mean_amplitude": np.mean(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_median_amplitude": np.median(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_std_amplitude": np.std(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_max_amplitude": np.max(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_sum_amplitude": np.sum(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_mean_rise_time": np.mean(rise_times) if rise_times.size else 0.0,
            "eda_scr_std_rise_time": np.std(rise_times) if rise_times.size else 0.0,
            "eda_scr_mean_recovery_time": np.mean(recovery_times) if recovery_times.size else 0.0,
            "eda_scr_std_recovery_time": np.std(recovery_times) if recovery_times.size else 0.0,
            "eda_phasic_auc": trapezoid(np.abs(phasic), dx=1 / sampling_rate),
            "eda_tonic_phasic_ratio": np.mean(np.abs(tonic)) / (np.mean(np.abs(phasic)) + EPSILON),
        })

        if peaks.size and onsets.size:
            n = min(peaks.size, onsets.size)
            widths = (peaks[:n] - onsets[:n]) / sampling_rate
            widths = widths[widths >= 0]
            out["eda_scr_mean_onset_to_peak"] = np.mean(widths) if widths.size else 0.0
            out["eda_scr_std_onset_to_peak"] = np.std(widths) if widths.size else 0.0

        return {k: safe_float(v) for k, v in out.items()}

    except Exception:
        return {k: safe_float(v) for k, v in out.items()}

## EMG

Las features de EMG incluyen amplitud, activación, complejidad y distribución espectral.

In [10]:
def extract_emg_features(emg: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    emg = sanitize_signal(emg)

    out = {}
    out.update(general_features(emg, sampling_rate, prefix="emg_"))

    tkeo_mean, tkeo_std = teager_kaiser_energy(emg)
    diff = np.diff(emg)

    out.update({
        "emg_mav": np.mean(np.abs(emg)),
        "emg_iemg": np.sum(np.abs(emg)) / sampling_rate,
        "emg_waveform_length": waveform_length(emg),
        "emg_zero_crossings_count": np.sum((emg[:-1] * emg[1:]) < 0),
        "emg_slope_sign_changes": slope_sign_changes(emg),
        "emg_willison_amplitude": willison_amplitude(emg),
        "emg_tkeo_mean": tkeo_mean,
        "emg_tkeo_std": tkeo_std,
        "emg_diff_mean_abs": np.mean(np.abs(diff)) if diff.size else 0.0,
        "emg_diff_std": np.std(diff) if diff.size else 0.0,
        "emg_log_detector": np.exp(np.mean(np.log(np.abs(emg) + EPSILON))),
    })

    out.update(spectral_features(
        emg,
        sampling_rate,
        prefix="emg_",
        bands={
            "low": (10.0, 60.0),
            "mid": (60.0, 150.0),
            "high": (150.0, 300.0),
        }
    ))

    out.update(nonlinear_features(emg, prefix="emg_"))
    out.update(wavelet_features(emg, prefix="emg_"))

    return {k: safe_float(v) for k, v in out.items()}

## Respiración

Se detectan picos y valles para aproximar ciclos respiratorios.  
De ellos se derivan frecuencia, duración, inspiración, espiración y variabilidad.

In [11]:
def extract_resp_features(resp: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    resp = sanitize_signal(resp)

    out = {}
    out.update(general_features(resp, sampling_rate, prefix="resp_"))
    out.update(spectral_features(
        resp,
        sampling_rate,
        prefix="resp_",
        bands={
            "very_slow": (0.0, 0.1),
            "normal_breathing": (0.1, 0.5),
            "fast_breathing": (0.5, 1.0),
        }
    ))
    out.update(nonlinear_features(resp, prefix="resp_"))
    out.update(wavelet_features(resp, prefix="resp_"))

    try:
        lowpassed = signal.savgol_filter(
            resp,
            window_length=min(701, len(resp) // 2 * 2 - 1),
            polyorder=3
        ) if len(resp) >= 11 else resp

        min_distance = int(sampling_rate * 1.0)
        peaks, _ = signal.find_peaks(lowpassed, distance=min_distance)
        troughs, _ = signal.find_peaks(-lowpassed, distance=min_distance)

        duration_min = resp.size / sampling_rate / 60.0
        out["resp_peak_count"] = float(peaks.size)
        out["resp_rate_per_minute"] = peaks.size / (duration_min + EPSILON)

        if peaks.size >= 2:
            cycle = np.diff(peaks) / sampling_rate
            out.update({
                "resp_cycle_mean_duration": np.mean(cycle),
                "resp_cycle_median_duration": np.median(cycle),
                "resp_cycle_std_duration": np.std(cycle),
                "resp_cycle_cv": np.std(cycle) / (np.mean(cycle) + EPSILON),
                "resp_breath_to_breath_rmssd": np.sqrt(np.mean(np.diff(cycle) ** 2))
                if cycle.size >= 2 else 0.0,
            })

        insp = []
        exp = []

        for p in peaks:
            previous_troughs = troughs[troughs < p]
            next_troughs = troughs[troughs > p]

            if previous_troughs.size:
                insp.append((p - previous_troughs[-1]) / sampling_rate)

            if next_troughs.size:
                exp.append((next_troughs[0] - p) / sampling_rate)

        insp = np.asarray(insp)
        exp = np.asarray(exp)

        out.update({
            "resp_inspiration_mean_duration": np.mean(insp) if insp.size else np.nan,
            "resp_inspiration_std_duration": np.std(insp) if insp.size else np.nan,
            "resp_expiration_mean_duration": np.mean(exp) if exp.size else np.nan,
            "resp_expiration_std_duration": np.std(exp) if exp.size else np.nan,
            "resp_inspiration_expiration_ratio": (
                np.mean(insp) / (np.mean(exp) + EPSILON)
                if insp.size and exp.size else np.nan
            ),
        })

    except Exception:
        pass

    return {k: safe_float(v) for k, v in out.items()}

## Temperatura

La temperatura cambia lentamente. Por eso son especialmente importantes:

- pendiente;
- delta;
- derivadas;
- curvatura;
- variabilidad local.

In [12]:
def extract_temp_features(temp: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    temp = sanitize_signal(temp)

    out = {}
    out.update(general_features(temp, sampling_rate, prefix="temp_"))

    first_derivative = np.gradient(temp) * sampling_rate
    second_derivative = np.gradient(first_derivative) * sampling_rate

    out.update({
        "temp_first_derivative_mean": np.mean(first_derivative),
        "temp_first_derivative_std": np.std(first_derivative),
        "temp_first_derivative_max_abs": np.max(np.abs(first_derivative)),
        "temp_second_derivative_mean": np.mean(second_derivative),
        "temp_second_derivative_std": np.std(second_derivative),
        "temp_curvature_mean_abs": np.mean(np.abs(second_derivative)),
        "temp_local_variability": np.mean(np.abs(np.diff(temp))) if temp.size > 1 else 0.0,
    })

    out.update(spectral_features(
        temp,
        sampling_rate,
        prefix="temp_",
        bands={
            "ultra_slow": (0.0, 0.01),
            "slow": (0.01, 0.1),
            "fast": (0.1, 1.0),
        }
    ))
    out.update(nonlinear_features(temp, prefix="temp_"))
    out.update(wavelet_features(temp, prefix="temp_"))

    return {k: safe_float(v) for k, v in out.items()}

## Acelerómetro

Las features se calculan para:

- X;
- Y;
- Z;
- magnitud vectorial;
- jerk;
- relaciones entre ejes.

In [13]:
def extract_acc_features(acc: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    acc = np.asarray(acc, dtype=np.float64)

    if acc.ndim != 2 or acc.shape[1] != 3:
        raise ValueError(f"ACC debe tener forma (n_muestras, 3), recibió {acc.shape}")

    x = sanitize_signal(acc[:, 0])
    y = sanitize_signal(acc[:, 1])
    z = sanitize_signal(acc[:, 2])
    magnitude = np.sqrt(x ** 2 + y ** 2 + z ** 2)

    out = {}

    for axis_name, axis_values in {
        "x": x,
        "y": y,
        "z": z,
        "magnitude": magnitude,
    }.items():
        prefix = f"acc_{axis_name}_"
        out.update(general_features(axis_values, sampling_rate, prefix=prefix))
        out.update(spectral_features(
            axis_values,
            sampling_rate,
            prefix=prefix,
            bands={
                "postural": (0.0, 0.5),
                "slow_motion": (0.5, 3.0),
                "fast_motion": (3.0, 10.0),
                "high_motion": (10.0, 20.0),
            }
        ))
        out.update(nonlinear_features(axis_values, prefix=prefix))
        out.update(wavelet_features(axis_values, prefix=prefix))

    jerk_x = np.gradient(x) * sampling_rate
    jerk_y = np.gradient(y) * sampling_rate
    jerk_z = np.gradient(z) * sampling_rate
    jerk_mag = np.sqrt(jerk_x ** 2 + jerk_y ** 2 + jerk_z ** 2)

    sma = np.mean(np.abs(x) + np.abs(y) + np.abs(z))
    svm = np.mean(magnitude)

    out.update({
        "acc_sma": sma,
        "acc_svm_mean": svm,
        "acc_svm_std": np.std(magnitude),
        "acc_jerk_mean": np.mean(jerk_mag),
        "acc_jerk_std": np.std(jerk_mag),
        "acc_jerk_max": np.max(jerk_mag),
        "acc_corr_xy": np.corrcoef(x, y)[0, 1] if np.std(x) > 0 and np.std(y) > 0 else 0.0,
        "acc_corr_xz": np.corrcoef(x, z)[0, 1] if np.std(x) > 0 and np.std(z) > 0 else 0.0,
        "acc_corr_yz": np.corrcoef(y, z)[0, 1] if np.std(y) > 0 and np.std(z) > 0 else 0.0,
        "acc_tilt_x_mean": np.mean(np.arctan2(x, np.sqrt(y ** 2 + z ** 2) + EPSILON)),
        "acc_tilt_y_mean": np.mean(np.arctan2(y, np.sqrt(x ** 2 + z ** 2) + EPSILON)),
        "acc_tilt_z_mean": np.mean(np.arctan2(z, np.sqrt(x ** 2 + y ** 2) + EPSILON)),
    })

    return {k: safe_float(v) for k, v in out.items()}

## Features multimodales

Estas features describen relaciones entre señales, no solo cada sensor por separado.

Para comparar señales con escalas diferentes se estandarizan primero.

In [14]:
def normalized_series(x: np.ndarray, max_points: int = NONLINEAR_MAX_POINTS) -> np.ndarray:
    x = downsample_for_nonlinear(x, max_points=max_points)
    return (x - np.mean(x)) / (np.std(x) + EPSILON)


def mutual_information_hist(x: np.ndarray, y: np.ndarray, bins: int = MUTUAL_INFO_BINS) -> float:
    x = normalized_series(x)
    y = normalized_series(y)

    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    joint_hist, _, _ = np.histogram2d(x, y, bins=bins)
    joint_prob = joint_hist / (np.sum(joint_hist) + EPSILON)
    px = np.sum(joint_prob, axis=1)
    py = np.sum(joint_prob, axis=0)

    expected = px[:, None] * py[None, :]
    mask = joint_prob > 0

    return safe_float(
        np.sum(joint_prob[mask] * np.log((joint_prob[mask] + EPSILON) / (expected[mask] + EPSILON))),
        np.nan
    )


def max_lagged_correlation(
    x: np.ndarray,
    y: np.ndarray,
    max_lag: int = 50
) -> Tuple[float, int]:
    x = normalized_series(x)
    y = normalized_series(y)

    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    best_corr = 0.0
    best_lag = 0

    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            a, b = x[-lag:], y[:n + lag]
        elif lag > 0:
            a, b = x[:n - lag], y[lag:]
        else:
            a, b = x, y

        if len(a) < 5:
            continue

        corr = np.corrcoef(a, b)[0, 1]
        if np.isfinite(corr) and abs(corr) > abs(best_corr):
            best_corr = corr
            best_lag = lag

    return safe_float(best_corr, 0.0), int(best_lag)


def coherence_summary(
    x: np.ndarray,
    y: np.ndarray,
    sampling_rate: float
) -> Tuple[float, float]:
    x = normalized_series(x)
    y = normalized_series(y)

    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    if n < 32:
        return np.nan, np.nan

    f, coh = signal.coherence(
        x,
        y,
        fs=sampling_rate,
        nperseg=min(256, n)
    )

    return safe_float(np.mean(coh)), safe_float(f[np.argmax(coh)])


def extract_multimodal_features(
    window_signals: Dict[str, np.ndarray],
    sampling_rate: float
) -> Dict[str, float]:
    if not ENABLE_MULTIMODAL:
        return {}

    acc = np.asarray(window_signals["ACC"], dtype=np.float64)
    acc_mag = np.sqrt(np.sum(acc ** 2, axis=1))

    one_d = {
        "ECG": sanitize_signal(window_signals["ECG"]),
        "EDA": sanitize_signal(window_signals["EDA"]),
        "EMG": sanitize_signal(window_signals["EMG"]),
        "Temp": sanitize_signal(window_signals["Temp"]),
        "Resp": sanitize_signal(window_signals["Resp"]),
        "ACCmag": sanitize_signal(acc_mag),
    }

    pairs = [
        ("ECG", "EDA"),
        ("ECG", "Resp"),
        ("EDA", "Resp"),
        ("EMG", "ACCmag"),
        ("Temp", "EDA"),
        ("Resp", "ACCmag"),
    ]

    out = {}

    for a_name, b_name in pairs:
        a = one_d[a_name]
        b = one_d[b_name]

        a_norm = normalized_series(a)
        b_norm = normalized_series(b)
        n = min(len(a_norm), len(b_norm))
        a_norm, b_norm = a_norm[:n], b_norm[:n]

        pair_prefix = f"multi_{a_name.lower()}_{b_name.lower()}_"

        corr = (
            np.corrcoef(a_norm, b_norm)[0, 1]
            if n > 2 and np.std(a_norm) > 0 and np.std(b_norm) > 0
            else 0.0
        )

        max_corr, lag = max_lagged_correlation(a_norm, b_norm, max_lag=30)
        coh_mean, coh_peak_freq = coherence_summary(a_norm, b_norm, sampling_rate)

        out.update({
            f"{pair_prefix}pearson": safe_float(corr, 0.0),
            f"{pair_prefix}mutual_information": mutual_information_hist(a_norm, b_norm),
            f"{pair_prefix}max_lagged_correlation": max_corr,
            f"{pair_prefix}best_lag_samples": lag,
            f"{pair_prefix}coherence_mean": coh_mean,
            f"{pair_prefix}coherence_peak_frequency": coh_peak_freq,
        })

    return out

## Registro de extractores

Cada extractor devuelve un diccionario `{nombre_feature: valor}`.  
El orden de columnas se fija con la primera ventana válida del sujeto y se conserva para todas las demás.

In [15]:
FEATURE_EXTRACTORS = {
    "ACC": extract_acc_features,
    "ECG": extract_ecg_features,
    "EMG": extract_emg_features,
    "EDA": extract_eda_features,
    "Temp": extract_temp_features,
    "Resp": extract_resp_features,
}

## Generación eficiente de ventanas

La función no crea una lista con todas las ventanas.

En su lugar:

1. identifica bloques contiguos con etiquetas válidas;
2. genera índices de inicio únicamente dentro de esos bloques;
3. toma una vista temporal de la ventana;
4. calcula features;
5. descarta la vista;
6. guarda solo el vector de features.

In [16]:
def contiguous_valid_blocks(
    labels: np.ndarray,
    valid_labels: Sequence[int]
) -> List[Tuple[int, int, int]]:
    labels = np.asarray(labels).reshape(-1)
    valid_set = set(valid_labels)

    blocks = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        value = int(labels[i])

        if value != current:
            if current in valid_set:
                blocks.append((start, i, current))
            start = i
            current = value

    if current in valid_set:
        blocks.append((start, len(labels), current))

    return blocks


def count_candidate_windows(
    blocks: List[Tuple[int, int, int]],
    window_samples: int,
    step_samples: int
) -> int:
    total = 0
    for start, end, _ in blocks:
        length = end - start
        if length >= window_samples:
            total += 1 + (length - window_samples) // step_samples
    return total

## Procesamiento de un sujeto

El resultado por sujeto contiene:

```python
{
    "feature_matrix": {
        "X": matriz float32,
        "names": nombres de columnas
    },
    "target": vector 0/1,
    "metadata": {...}
}
```

No se conservan segmentos crudos.

In [17]:
def process_single_subject(
    subject: str,
    base_path: Path = BASE_PATH,
    window_ms: float = WINDOW_MS,
    step_ms: float = STEP_MS,
    sampling_rate: float = SAMPLING_RATE,
    negative_labels: Sequence[int] = NEGATIVE_LABELS,
    positive_labels: Sequence[int] = POSITIVE_LABELS,
    selected_signals: Sequence[str] = SELECTED_SIGNALS,
    purity_threshold: float = PURITY_THRESHOLD,
) -> Dict:
    data = load_wesad_subject(base_path, subject)

    chest = data["signal"]["chest"]
    labels = np.asarray(data["label"]).reshape(-1)

    negative_set = set(negative_labels)
    positive_set = set(positive_labels)
    valid_labels = sorted(negative_set | positive_set)

    overlap = negative_set & positive_set
    if overlap:
        raise ValueError(f"Etiquetas repetidas en ambas clases: {sorted(overlap)}")

    window_samples = int(round(window_ms * sampling_rate / 1000))
    step_samples = int(round(step_ms * sampling_rate / 1000))

    signal_lengths = {
        name: np.asarray(chest[name]).shape[0]
        for name in selected_signals
    }

    minimum_length = min([len(labels)] + list(signal_lengths.values()))
    labels = labels[:minimum_length]

    blocks = contiguous_valid_blocks(labels, valid_labels)
    n_candidates = count_candidate_windows(blocks, window_samples, step_samples)

    feature_rows = []
    targets = []
    starts = []
    ends = []
    original_labels = []
    purities = []
    feature_names = None

    iterator = tqdm(total=n_candidates, desc=f"{subject}", leave=True)

    try:
        for block_start, block_end, original_label in blocks:
            block_length = block_end - block_start

            if block_length < window_samples:
                continue

            for start in range(
                block_start,
                block_end - window_samples + 1,
                step_samples
            ):
                end = start + window_samples
                iterator.update(1)

                window_labels = labels[start:end]
                purity = np.mean(window_labels == original_label)

                if purity < purity_threshold:
                    continue

                window_signals = {
                    name: np.asarray(chest[name])[start:end]
                    for name in selected_signals
                }

                row_dict = {}

                for signal_name in selected_signals:
                    extracted = FEATURE_EXTRACTORS[signal_name](
                        window_signals[signal_name],
                        sampling_rate=sampling_rate
                    )
                    row_dict.update(extracted)

                if ENABLE_MULTIMODAL:
                    row_dict.update(
                        extract_multimodal_features(
                            window_signals,
                            sampling_rate=sampling_rate
                        )
                    )

                if feature_names is None:
                    feature_names = list(row_dict.keys())

                row = [
                    safe_float(row_dict.get(name, np.nan))
                    for name in feature_names
                ]

                feature_rows.append(row)
                targets.append(0 if original_label in negative_set else 1)
                starts.append(start)
                ends.append(end)
                original_labels.append(original_label)
                purities.append(purity)

                del window_signals, row_dict, row

    finally:
        iterator.close()

    X = np.asarray(feature_rows, dtype=OUTPUT_DTYPE)
    y = np.asarray(targets, dtype=np.int8)

    result = {
        "subject": subject,
        "feature_matrix": {
            "X": X,
            "names": feature_names or []
        },
        "target": y,
        "metadata": {
            "window_ms": window_ms,
            "step_ms": step_ms,
            "sampling_rate": sampling_rate,
            "window_samples": window_samples,
            "step_samples": step_samples,
            "starts": np.asarray(starts, dtype=np.int64),
            "ends": np.asarray(ends, dtype=np.int64),
            "purity": np.asarray(purities, dtype=np.float32),
            "original_label": np.asarray(original_labels, dtype=np.int8),
            "n_segments": int(len(y)),
            "n_features": int(X.shape[1]) if X.ndim == 2 else 0,
            "label_mapping": {
                "negative_target_0": sorted(negative_set),
                "positive_target_1": sorted(positive_set),
                "excluded": sorted(
                    set(np.unique(labels)) - negative_set - positive_set
                ),
            },
            "config": CONFIG,
        }
    }

    del data, chest, labels, feature_rows, targets, starts, ends
    gc.collect()

    return result

## Guardado incremental y reanudación

Cada sujeto se guarda en:

```text
wesad_feature_output/checkpoints/S2_features.pkl
```

Si el proceso se interrumpe, la siguiente ejecución puede continuar desde los sujetos ya guardados.

In [18]:
def subject_checkpoint_path(subject: str) -> Path:
    return CHECKPOINT_DIR / f"{subject}_features.pkl"


def save_pickle_atomic(obj, path: Path) -> None:
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")

    with open(temporary, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

    os.replace(temporary, path)


def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)

## Procesar todos los sujetos

Esta celda:

- procesa un sujeto;
- guarda el checkpoint;
- elimina el objeto de memoria;
- ejecuta el recolector de basura;
- continúa con el siguiente.

Durante esta etapa no se mantiene el diccionario completo en RAM.

In [19]:
def process_all_subjects_incrementally(
    subjects: Sequence[str] = SUBJECTS
) -> Dict[str, str]:
    status = {}

    for subject in subjects:
        checkpoint = subject_checkpoint_path(subject)

        if (
            checkpoint.exists()
            and RESUME_FROM_CHECKPOINTS
            and not OVERWRITE_EXISTING_SUBJECTS
        ):
            print(f"{subject}: checkpoint existente, se omite.")
            status[subject] = "existing"
            continue

        try:
            print(f"\nProcesando {subject}...")
            result = process_single_subject(subject)
            save_pickle_atomic(result, checkpoint)

            print(
                f"{subject}: guardado | "
                f"ventanas={result['metadata']['n_segments']} | "
                f"features={result['metadata']['n_features']}"
            )

            status[subject] = "processed"

            del result
            gc.collect()

        except Exception as error:
            status[subject] = f"error: {type(error).__name__}: {error}"
            print(f"{subject}: ERROR -> {error}")
            gc.collect()

    return status


processing_status = process_all_subjects_incrementally()
processing_status


Procesando S3...


S3:   0%|          | 0/7903 [00:00<?, ?it/s]

S3: guardado | ventanas=7903 | features=869

Procesando S4...


S4:   0%|          | 0/7943 [00:00<?, ?it/s]

S4: guardado | ventanas=7943 | features=869

Procesando S5...


...

S16:   0%|          | 0/8167 [00:00<?, ?it/s]

S16: guardado | ventanas=8167 | features=869

Procesando S17...


S17:   0%|          | 0/8387 [00:00<?, ?it/s]

S17: guardado | ventanas=8387 | features=869


{'S3': 'processed',
 'S4': 'processed',
 'S5': 'processed',
 'S6': 'processed',
 'S7': 'processed',
 'S8': 'processed',
 'S9': 'processed',
 'S10': 'processed',
 'S11': 'processed',
 'S13': 'processed',
 'S14': 'processed',
 'S15': 'processed',
 'S16': 'processed',
 'S17': 'processed'}

# Consolidación final

La siguiente función carga los checkpoints uno por uno y construye el diccionario final solicitado:

```python
wesad_processed["S2"]
wesad_processed["S3"]
...
```

En equipos con muy poca memoria se puede trabajar directamente con los checkpoints sin consolidarlos.  
La consolidación final contiene únicamente features y metadatos, no las señales crudas.

In [27]:
def consolidate_checkpoints(
    subjects: Sequence[str] = SUBJECTS,
    output_path: Path = FINAL_OUTPUT_PATH
) -> Dict:
    wesad_processed = {
        "_global_metadata": {
            "config": CONFIG,
            "subjects_requested": list(subjects),
            "subjects_available": []
        }
    }

    for subject in subjects:
        checkpoint = subject_checkpoint_path(subject)

        if not checkpoint.exists():
            print(f"Advertencia: no existe checkpoint para {subject}")
            continue

        subject_result = load_pickle(checkpoint)
        wesad_processed[subject] = subject_result
        wesad_processed["_global_metadata"]["subjects_available"].append(subject)

    save_pickle_atomic(wesad_processed, output_path)

    print(f"Archivo final guardado en:\n{output_path}")
    print(
        "Sujetos consolidados:",
        wesad_processed["_global_metadata"]["subjects_available"]
    )

    return wesad_processed


wesad_processed = consolidate_checkpoints()

Archivo final guardado en:
C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\wesad_processed_full_features.pkl
Sujetos consolidados: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']


## Inspección del resultado

Comprueba:

- forma de la matriz;
- número de targets;
- balance de clases;
- nombres de features;
- ausencia de segmentos crudos.

In [28]:
def summarize_processed_dictionary(wesad_processed: Dict) -> pd.DataFrame:
    rows = []

    for subject, data in wesad_processed.items():
        if subject.startswith("_"):
            continue

        X = data["feature_matrix"]["X"]
        y = data["target"]

        rows.append({
            "subject": subject,
            "n_windows": len(y),
            "n_features": X.shape[1] if X.ndim == 2 else 0,
            "class_0": int(np.sum(y == 0)),
            "class_1": int(np.sum(y == 1)),
            "memory_mb": round(X.nbytes / (1024 ** 2), 2)
        })

    return pd.DataFrame(rows)


summary_df = summarize_processed_dictionary(wesad_processed)
summary_df

,subject,n_windows,n_features,class_0,class_1,memory_mb
0,S2,7767,869,5546,2221,25.75
1,S3,7903,869,5582,2321,26.20
2,S4,7943,869,5642,2301,26.33
3,S5,8151,869,5810,2341,27.02
4,S6,8091,869,5730,2361,26.82
5,S7,8075,869,5754,2321,26.77
6,S8,8118,869,5677,2441,26.91
7,S9,8071,869,5730,2341,26.76
8,S10,8391,869,5730,2661,27.82
9,S11,8195,869,5714,2481,27.17


In [29]:
# Ejemplo de inspección detallada
subject = "S2"

X = wesad_processed[subject]["feature_matrix"]["X"]
y = wesad_processed[subject]["target"]
feature_names = wesad_processed[subject]["feature_matrix"]["names"]

print("X:", X.shape)
print("y:", y.shape)
print("Clases:", np.unique(y, return_counts=True))
print("Primeras 20 features:")
print(feature_names[:20])

X: (7767, 869)
y: (7767,)
Clases: (array([0, 1], dtype=int8), array([5546, 2221]))
Primeras 20 features:
['acc_x_mean', 'acc_x_median', 'acc_x_std', 'acc_x_variance', 'acc_x_min', 'acc_x_max', 'acc_x_range', 'acc_x_q01', 'acc_x_q05', 'acc_x_q10', 'acc_x_q25', 'acc_x_q75', 'acc_x_q90', 'acc_x_q95', 'acc_x_q99', 'acc_x_iqr', 'acc_x_mad_mean', 'acc_x_mad_median', 'acc_x_rms', 'acc_x_mean_absolute_value']


## Cargar el archivo final en una sesión futura

In [30]:
import pickle
from pathlib import Path

FINAL_OUTPUT_PATH = Path.cwd() / "wesad_feature_output" / "wesad_processed_full_features.pkl"

with open(FINAL_OUTPUT_PATH, "rb") as f:
    wesad_processed = pickle.load(f)

print(
    "Sujetos disponibles:",
    wesad_processed["_global_metadata"]["subjects_available"]
)

Sujetos disponibles: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']


## Cargar solo un sujeto para ahorrar memoria

En un equipo de baja potencia es preferible abrir únicamente el checkpoint necesario.

In [31]:
def load_processed_subject(subject: str) -> Dict:
    checkpoint = Path.cwd() / "wesad_feature_output" / "checkpoints" / f"{subject}_features.pkl"

    with open(checkpoint, "rb") as f:
        return pickle.load(f)


s2_processed = load_processed_subject("S2")

print(s2_processed["feature_matrix"]["X"].shape)
print(s2_processed["target"].shape)

# Liberar cuando ya no sea necesario:
# del s2_processed
# gc.collect()

(7767, 869)
(7767,)


## Crear una tabla automática de features

La tabla se construye a partir de los nombres reales generados por la notebook.  
Esto evita diferencias entre la documentación y el código.

In [32]:
def classify_feature_family(name: str) -> str:
    if name.startswith("acc_"):
        return "ACC"
    if name.startswith("ecg_"):
        return "ECG"
    if name.startswith("emg_"):
        return "EMG"
    if name.startswith("eda_"):
        return "EDA"
    if name.startswith("temp_"):
        return "Temp"
    if name.startswith("resp_"):
        return "Resp"
    if name.startswith("multi_"):
        return "Multimodal"
    return "Other"


def describe_feature(name: str) -> str:
    rules = [
        ("mean", "Promedio del indicador dentro de la ventana."),
        ("median", "Mediana; medida robusta del nivel central."),
        ("std", "Desviación estándar; cuantifica variabilidad."),
        ("variance", "Varianza; dispersión cuadrática."),
        ("min", "Valor mínimo dentro de la ventana."),
        ("max", "Valor máximo dentro de la ventana."),
        ("range", "Diferencia entre máximo y mínimo."),
        ("iqr", "Rango intercuartílico; dispersión robusta."),
        ("rms", "Valor cuadrático medio; amplitud o energía efectiva."),
        ("energy", "Energía acumulada de la señal o coeficientes."),
        ("entropy", "Irregularidad o complejidad de la señal."),
        ("dfa", "Dependencia de largo alcance mediante análisis de fluctuaciones sin tendencia."),
        ("higuchi_fd", "Dimensión fractal de Higuchi."),
        ("katz_fd", "Dimensión fractal de Katz."),
        ("petrosian_fd", "Dimensión fractal de Petrosian."),
        ("rqa_", "Métrica de recurrencia de la dinámica de la señal."),
        ("wavelet", "Característica derivada de coeficientes wavelet."),
        ("dominant_frequency", "Frecuencia con mayor potencia espectral."),
        ("mean_frequency", "Promedio ponderado de frecuencias."),
        ("median_frequency", "Frecuencia que divide la potencia espectral en dos mitades."),
        ("spectral", "Descriptor de la distribución de potencia en frecuencia."),
        ("slope", "Tendencia lineal durante la ventana."),
        ("delta", "Cambio entre el final y el inicio."),
        ("correlation", "Asociación lineal entre dos señales o ejes."),
        ("mutual_information", "Dependencia no lineal entre dos señales."),
        ("coherence", "Acoplamiento lineal en el dominio de la frecuencia."),
        ("rmssd", "Variabilidad de corto plazo entre intervalos consecutivos."),
        ("sdnn", "Variabilidad global de los intervalos NN."),
        ("pnn20", "Porcentaje de diferencias NN superiores a 20 ms."),
        ("pnn50", "Porcentaje de diferencias NN superiores a 50 ms."),
        ("scr", "Característica de las respuestas fásicas de conductancia."),
        ("resp_rate", "Frecuencia respiratoria estimada."),
        ("jerk", "Derivada temporal de la aceleración."),
        ("sma", "Área de magnitud de señal del acelerómetro."),
        ("tkeo", "Energía instantánea de Teager–Kaiser."),
        ("willison", "Conteo de cambios de amplitud EMG superiores a un umbral."),
        ("waveform_length", "Longitud acumulada de la trayectoria de la señal."),
    ]

    lower = name.lower()

    for token, description in rules:
        if token in lower:
            return description

    return "Descriptor calculado sobre la ventana de la señal correspondiente."


def build_feature_catalog(feature_names: Sequence[str]) -> pd.DataFrame:
    return pd.DataFrame({
        "signal": [classify_feature_family(name) for name in feature_names],
        "feature": list(feature_names),
        "meaning": [describe_feature(name) for name in feature_names]
    }).sort_values(["signal", "feature"]).reset_index(drop=True)


feature_catalog = build_feature_catalog(
    wesad_processed["S2"]["feature_matrix"]["names"]
)

feature_catalog

,signal,feature,meaning
0,ACC,acc_corr_xy,Descriptor calculado sobre la ventana de la se...
1,ACC,acc_corr_xz,Descriptor calculado sobre la ventana de la se...
2,ACC,acc_corr_yz,Descriptor calculado sobre la ventana de la se...
3,ACC,acc_jerk_max,Valor máximo dentro de la ventana.
4,ACC,acc_jerk_mean,Promedio del indicador dentro de la ventana.
...,...,...,...
864,Temp,temp_wavelet_D4_entropy,Irregularidad o complejidad de la señal.
865,Temp,temp_wavelet_D4_mean_abs,Promedio del indicador dentro de la ventana.
866,Temp,temp_wavelet_D4_relative_energy,Energía acumulada de la señal o coeficientes.
867,Temp,temp_wavelet_D4_std,Desviación estándar; cuantifica variabilidad.


In [36]:
print(feature_catalog.to_markdown())

|     | signal     | feature                                    | meaning                                                                        |
|----:|:-----------|:-------------------------------------------|:-------------------------------------------------------------------------------|
|   0 | ACC        | acc_corr_xy                                | Descriptor calculado sobre la ventana de la señal correspondiente.             |
|   1 | ACC        | acc_corr_xz                                | Descriptor calculado sobre la ventana de la señal correspondiente.             |
|   2 | ACC        | acc_corr_yz                                | Descriptor calculado sobre la ventana de la señal correspondiente.             |
|   3 | ACC        | acc_jerk_max                               | Valor máximo dentro de la ventana.                                             |
|   4 | ACC        | acc_jerk_mean                              | Promedio del indicador dentro de la ventana.        

## Exportar el catálogo de features

El catálogo se puede guardar en CSV para anexarlo al informe metodológico.

In [33]:
FEATURE_CATALOG_PATH = OUTPUT_DIR / "feature_catalog.csv"
feature_catalog.to_csv(FEATURE_CATALOG_PATH, index=False, encoding="utf-8-sig")

print(FEATURE_CATALOG_PATH)

C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\feature_catalog.csv


## Recomendaciones metodológicas finales

1. **No realizar una división aleatoria de ventanas** cuando existen solapamientos altos. Las ventanas adyacentes son casi idénticas y pueden producir fuga de información.
2. Utilizar validación por sujeto, preferiblemente **LOSO**.
3. Ajustar imputación, normalización y selección de features solamente con los sujetos de entrenamiento.
4. Registrar exactamente:
   - ventana;
   - desplazamiento;
   - etiquetas incluidas;
   - etiquetas excluidas;
   - pureza;
   - features habilitadas;
   - versión de librerías.
5. Ejecutar primero con `STEP_MS = 5000` y pocos sujetos.
6. Cambiar a `STEP_MS = 250` únicamente después de verificar estabilidad y tiempo de procesamiento.
7. Las features no lineales y RQA son las más costosas. La notebook las incluye, pero están diseñadas con límites internos para no agotar la memoria.
8. Los valores `NaN` deben imputarse dentro del pipeline de validación, nunca usando información del sujeto de prueba.